# Применение ансамблей моделей 

http://xgboost.readthedocs.io/en/latest/  
http://xgboost.readthedocs.io/en/latest/model.html  
https://lightgbm.readthedocs.io/  
https://lightgbm.readthedocs.io/en/latest/    
https://tech.yandex.com/catboost/doc/dg/concepts/about-docpage/   
http://docs.h2o.ai/h2o/latest-stable/h2o-docs/data-science/gbm.html#defining-a-gbm-model  

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import time
import re
from __future__ import print_function
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_union, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, LabelEncoder, MinMaxScaler, LabelBinarizer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV

import xgboost as xgb
import lightgbm as lgb

%matplotlib inline
plt.rcParams["figure.figsize"] = (15, 8)
pd.options.display.float_format = '{:.2f}'.format

In [3]:
df_train = pd.read_csv('data/train.csv')
df_train.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.00,1,0,A/5 21171,7.25,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.00,1,0,PC 17599,71.28,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.00,0,0,STON/O2. 3101282,7.92,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.00,1,0,113803,53.10,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.00,0,0,373450,8.05,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.46,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.00,0,0,17463,51.86,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.00,3,1,349909,21.07,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.00,0,2,347742,11.13,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.00,1,0,237736,30.07,NaN,C


In [4]:
df_test = pd.read_csv('data/test.csv')
df_test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.50,0,0,330911,7.83,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.00,1,0,363272,7.00,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.00,0,0,240276,9.69,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.00,0,0,315154,8.66,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.00,1,1,3101298,12.29,NaN,S


In [5]:
# move target to the right
survived = df_train['Survived']
df_train.drop(labels=['Survived'], axis=1, inplace=True)
df_train['Survived'] = survived
df_train.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,1,3,"Braund, Mr. Owen Harris",male,22.00,1,0,A/5 21171,7.25,NaN,S,0
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.00,1,0,PC 17599,71.28,C85,C,1
2,3,3,"Heikkinen, Miss. Laina",female,26.00,0,0,STON/O2. 3101282,7.92,NaN,S,1
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.00,1,0,113803,53.10,C123,S,1
4,5,3,"Allen, Mr. William Henry",male,35.00,0,0,373450,8.05,NaN,S,0


In [6]:
class LabelEncoderPipelineFriendly(LabelEncoder):
    
    def fit(self, X, y=None):
        super(LabelEncoderPipelineFriendly, self).fit(X)
        
    def transform(self, X, y=None):
        return super(LabelEncoderPipelineFriendly, self).transform(X).reshape(-1, 1)

    def fit_transform(self, X, y=None):
        return super(LabelEncoderPipelineFriendly, self).fit(X).transform(X).reshape(-1, 1)
    

class FeaturesSum(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self
        
    def transform(self, X, y=None):
        return np.sum(X, axis=1).reshape(-1, 1)

    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)
    

class AgeFeature(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self
        
    def transform(self, X, y=None):
        X['Initial'] = 0
        for i in X:
            X['Initial'] = X.Name.str.extract('([A-Za-z]+)\.') 
    
        X['Initial'].replace(
            ['Mlle','Mme','Ms','Dr','Major','Lady','Countess','Jonkheer','Col','Rev','Capt','Sir','Don'],
            ['Miss','Miss','Miss','Mr','Mr','Mrs','Mrs','Other','Other','Other','Mr','Mr','Mr'],
            inplace=True
        )
        X.groupby('Initial')['Age'].mean() 

        X.loc[(X.Age.isnull()) & (X.Initial=='Mr'), 'Age'] = 33
        X.loc[(X.Age.isnull()) & (X.Initial=='Mrs'), 'Age'] = 36
        X.loc[(X.Age.isnull()) & (X.Initial=='Master'), 'Age'] = 5
        X.loc[(X.Age.isnull()) & (X.Initial=='Miss'), 'Age'] = 22
        X.loc[(X.Age.isnull()) & (X.Initial=='Other'), 'Age'] = 46
        return X['Age'].to_numpy().reshape(-1, 1)

    def fit_transform(self, X, y=None):
        return self.fit(X).transform(X)

In [7]:
def get_sex_col(df):
    return df[['Sex']]

def get_age_name_cols(df):
    return df[['Age', 'Name']]

def get_pclass_col(df):
    return df[['Pclass']]

def get_sum_cols(df):
    return df[['Age', 'Fare']]

def get_num_cols(df):
    return df[['Fare', 'SibSp', 'Parch']]

vec = make_union(*[
    make_pipeline(FunctionTransformer(get_pclass_col, validate=False),  OneHotEncoder(sparse_output=False)),
    make_pipeline(FunctionTransformer(get_sex_col, validate=False),  LabelEncoderPipelineFriendly()),
    make_pipeline(FunctionTransformer(get_num_cols, validate=False), SimpleImputer(strategy='mean'), MinMaxScaler()),
    make_pipeline(FunctionTransformer(get_age_name_cols, validate=False),  AgeFeature()),
])

In [8]:
x_train = vec.fit_transform(df_train)
x_train.shape

(891, 8)

In [9]:
x_train

array([[ 0.        ,  0.        ,  1.        , ...,  0.125     ,
         0.        , 22.        ],
       [ 1.        ,  0.        ,  0.        , ...,  0.125     ,
         0.        , 38.        ],
       [ 0.        ,  0.        ,  1.        , ...,  0.        ,
         0.        , 26.        ],
       ...,
       [ 0.        ,  0.        ,  1.        , ...,  0.125     ,
         0.33333333, 22.        ],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , 26.        ],
       [ 0.        ,  0.        ,  1.        , ...,  0.        ,
         0.        , 32.        ]], shape=(891, 8))

In [10]:
y_train = df_train['Survived']
y_train.shape

(891,)

In [11]:
lr = LogisticRegressionCV(cv=10)
lr.fit(x_train, y_train)
lr

,Cs,10
,fit_intercept,True
,cv,10
,dual,False
,penalty,'l2'
,scoring,None
,solver,'lbfgs'
,tol,0.0001
,max_iter,100
,class_weight,None
,n_jobs,None


In [12]:
accuracy_score(y_train, lr.predict(x_train))

0.8035914702581369

# Применение модели

In [13]:
def apply_model(model, submission_name):
    x_test = vec.fit_transform(df_test) 
    print('shape of x_test is {}'.format(x_test.shape))
    y_test = model.predict(x_test)
    print('shape of y_test is {}'.format(y_test.shape))
    df_predicted = pd.DataFrame({'PassengerId': df_test['PassengerId'], 'Survived': y_test})
    df_predicted.to_csv(submission_name + '.csv', sep=',', index=False)

In [14]:
apply_model(lr, 'log_regression_cv')

shape of x_test is (418, 8)
shape of y_test is (418,)


# Обучение ансамблей

In [15]:
def randomized_cv(model, param_grid, x_train=x_train, y_train=y_train):
    grid_search = RandomizedSearchCV(model, param_grid, cv=5, scoring='accuracy')
    t_start = time.time()
    grid_search.fit(x_train, y_train)
    t_end = time.time()
    print('model {} best accuracy score is {}'.format(model.__class__.__name__, grid_search.best_score_))
    print('time for training is {} seconds'.format(t_end - t_start))
    print(grid_search.best_score_)
    return grid_search.best_estimator_

# XGBoost

In [16]:
import xgboost as xgb

param_grid = {
    'max_depth': [2, 3, 4],
    'n_estimators': [50, 100],
    'learning_rate': [0.01, 0.025]
}
xgb = randomized_cv(xgb.XGBClassifier(), param_grid)

model XGBClassifier best accuracy score is 0.8249074132195091
time for training is 1.866668462753296 seconds
0.8249074132195091


In [17]:
apply_model(xgb, 'xgb_cv')

shape of x_test is (418, 8)
shape of y_test is (418,)


Особенности XGBoost
* есть регуляризация
* распараллеливание
* возможность кастомизации
* обработка отсуствующих значений
* встроенная кросс-валидация
* возможность архивировать и восстанавливать модель

# LightGBM 

In [18]:
import lightgbm as lgb

param_grid = {
    'max_depth': [2, 3, 4, 5],
    'n_estimators': [50, 100, 150, 200],
    'learning_rate': [0.01, 0.02, 0.05]
}
model = randomized_cv(lgb.LGBMClassifier(), param_grid)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000164 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 201
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


model LGBMClassifier best accuracy score is 0.8260435628648548

time for training is 3.5334842205047607 seconds

0.8260435628648548

In [19]:
apply_model(model, 'lgb_cv')

shape of x_test is (418, 8)
shape of y_test is (418,)


Особенности

* использование гистограмм для всех признаков (уже тоже есть в xgboost)
* то же самое, но быстрее (см выше)

# H2O GBM 

In [20]:
import h2o
import numpy as np
import math
from h2o.estimators.gbm import H2OGradientBoostingEstimator
from h2o.grid.grid_search import H2OGridSearch
h2o.init(nthreads=-1, strict_version_check=True)

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 25.51-b03, mixed mode)
  Starting server from C:\Users\admin\anaconda3\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\admin\AppData\Local\Temp\tmpwoapuyyt
  JVM stdout: C:\Users\admin\AppData\Local\Temp\tmpwoapuyyt\h2o_admin_started_from_python.out
  JVM stderr: C:\Users\admin\AppData\Local\Temp\tmpwoapuyyt\h2o_admin_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,Europe/Moscow
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,1 month and 22 days
H2O_cluster_name:,H2O_from_python_admin_eur99z
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,7.044 Gb
H2O_cluster_total_cores:,16
H2O_cluster_allowed_cores:,16
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+


In [22]:
data = np.c_[x_train, y_train]
data = pd.DataFrame(data, columns=['C{}'.format(idx + 1) for idx in range(data.shape[-1])])

train_df_h2o = h2o.H2OFrame(python_obj=data)
train_df_h2o['C9'] = train_df_h2o['C9'].asfactor()

train_df_h2o.show()

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


C1,C2,C3,C4,C5,C6,C7,C8,C9
0,0,1,1,0.0141511,0.125,0,22,0
1,0,0,0,0.139136,0.125,0,38,1
0,0,1,0,0.0154686,0,0,26,1
1,0,0,0,0.103644,0.125,0,35,1
0,0,1,1,0.0157126,0,0,35,0
0,0,1,1,0.0165095,0,0,33,0
1,0,0,1,0.101229,0,0,54,0
0,0,1,1,0.0411357,0.375,0.166667,2,0
0,0,1,0,0.0217308,0,0.333333,27,1
0,1,0,0,0.0586943,0.125,0,14,1


In [23]:
x_test = vec.fit_transform(df_test)
data_test = pd.DataFrame(x_test, columns=['C{}'.format(idx + 1) for idx in range(x_test.shape[-1])])

test_df_h2o = h2o.H2OFrame(python_obj=data_test)
test_df_h2o.show()

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


C1,C2,C3,C4,C5,C6,C7,C8
0,0,1,1,0.0152816,0,0,34.5
0,0,1,0,0.0136631,0.125,0,47
0,1,0,1,0.0189087,0,0,62
0,0,1,1,0.0169081,0,0,27
0,0,1,0,0.0239836,0.125,0.111111,22
0,0,1,1,0.018006,0,0,14
0,0,1,0,0.0148912,0,0,30
0,1,0,1,0.0566042,0.125,0.111111,26
0,0,1,0,0.0141105,0,0,18
0,0,1,1,0.0471377,0.25,0,21


In [24]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   C1      891 non-null    float64
 1   C2      891 non-null    float64
 2   C3      891 non-null    float64
 3   C4      891 non-null    float64
 4   C5      891 non-null    float64
 5   C6      891 non-null    float64
 6   C7      891 non-null    float64
 7   C8      891 non-null    float64
 8   C9      891 non-null    float64
dtypes: float64(9)
memory usage: 62.8 KB


In [25]:
gbm = H2OGradientBoostingEstimator()
gbm.train(x=['C{}'.format(idx + 1) for idx in range(data.shape[-1] - 1)], y='C9', training_frame=train_df_h2o)
print(gbm)

gbm Model Build progress: |██████████████████████████████████████████████████████| (done) 100%
Model Details
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_model_python_1791036099947_1


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    50                 50                          13290                  5            5            5             6             29            16.46

ModelMetricsBinomial: gbm
** Reported on train data. **

MSE: 0.0893923322821615
RMSE: 0.2989855051372248
LogLoss: 0.3044905775157067
Mean Per-Class Error: 0.11851692071709327
AUC: 0.9412461785915912
AUCPR: 0.9298919513771351
Gini: 0.8824923571831824

Confusion Matrix (Act/Pred) for max f1 @ threshold = 0.479021

In [27]:
hyper_params = {'max_depth' : list(range(1,30,2))}

gbm_grid = H2OGradientBoostingEstimator(
        ntrees=10000,
        learn_rate=0.05,
        learn_rate_annealing = 0.99,
        sample_rate = 0.8,
        col_sample_rate = 0.8,
        seed = 1234,
        score_tree_interval = 10, 
        stopping_rounds = 5,
        stopping_metric = "misclassification",
        stopping_tolerance = 1e-4)

grid = H2OGridSearch(gbm_grid, hyper_params,
                         grid_id = 'depth_grid',
                         search_criteria = {'strategy': "RandomDiscrete"})


grid.train(x=['C{}'.format(idx + 1) for idx in range(data.shape[-1] - 1)], y='C9', training_frame=train_df_h2o)

gbm Grid Build progress: |███████████████████████████████████████████████████████| (done) 100%


,max_depth,model_ids,logloss
,27.0,depth_grid_model_10,0.2029916
,29.0,depth_grid_model_5,0.2029916
,23.0,depth_grid_model_11,0.2039714
,25.0,depth_grid_model_4,0.2039818
,21.0,depth_grid_model_9,0.2043141
,19.0,depth_grid_model_12,0.2049349
,17.0,depth_grid_model_6,0.2058869
,15.0,depth_grid_model_13,0.2077973
,13.0,depth_grid_model_15,0.2157335
,11.0,depth_grid_model_7,0.2345442


In [28]:
print(grid)

Hyper-Parameter Search Summary: ordered by increasing logloss
    max_depth    model_ids            logloss
--  -----------  -------------------  ---------
    27           depth_grid_model_10  0.202992
    29           depth_grid_model_5   0.202992
    23           depth_grid_model_11  0.203971
    25           depth_grid_model_4   0.203982
    21           depth_grid_model_9   0.204314
    19           depth_grid_model_12  0.204935
    17           depth_grid_model_6   0.205887
    15           depth_grid_model_13  0.207797
    13           depth_grid_model_15  0.215733
    11           depth_grid_model_7   0.234544
    9            depth_grid_model_1   0.238381
    7            depth_grid_model_14  0.255465
    5            depth_grid_model_2   0.298966
    3            depth_grid_model_8   0.358717
    1            depth_grid_model_3   0.460685


In [29]:
best_model = h2o.get_model(grid.sorted_metric_table()['model_ids'][0])
best_model

Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: depth_grid_model_10


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    500                500                         358966                 9            27           16.272        46            57            52.058

ModelMetricsBinomial: gbm
** Reported on train data. **

MSE: 0.05619102209709977
RMSE: 0.2370464555674684
LogLoss: 0.20299158583991428
Mean Per-Class Error: 0.07427379925222893
AUC: 0.9820487009874413
AUCPR: 0.9747627645851848
Gini: 0.9640974019748827

Confusion Matrix (Act/Pred) for max f1 @ threshold = 0.4157814191724123
       0    1    Error    Rate
-----  ---  ---  -------  ------------
0      514  35   0.0638   (35.0/549.0)
1      29   313  0.0848   (29.0/342.0)
Total  543  348  0.0718   (64.0/891.0)

Maximum Metrics: Maximum metrics at their respective thresholds
metric                       threshold    value     idx
---------------------------  -----------  --------  -----
max f1                       0.415781     0.907246  193
max f2                       0.252425     0.939086  233
max f0point5                 0.610053     0.932983  153
max accuracy                 0.48189      0.929293  174
max precision                0.991773     1         0
max recall                   0.0692522    1         334
max specificity              0.991773     1         0
max absolute_mcc             0.445779     0.849968  183
max min_per_class_accuracy   0.361913     0.921676  200
max mean_per_class_accuracy  0.343391     0.925942  205
max tns                      0.991773     549       0
max fns                      0.991773     340       0
max fps                      0.00724691   549       399
max tps                      0.0692522    342       334
max tnr                      0.991773     1         0
max fnr                      0.991773     0.994152  0
max fpr                      0.00724691   1         399
max tpr                      0.0692522    1         334

Gains/Lift Table: Avg response rate: 38,38 %, avg score: 38,38 %
group    cumulative_data_fraction    lower_threshold    lift       cumulative_lift    response_rate    score      cumulative_response_rate    cumulative_score    capture_rate    cumulative_capture_rate    gain      cumulative_gain    kolmogorov_smirnov
-------  --------------------------  -----------------  ---------  -----------------  ---------------  ---------  --------------------------  ------------------  --------------  -------------------------  --------  -----------------  --------------------
1        0.010101                    0.989562           2.60526    2.60526            1                0.9906     1                           0.9906              0.0263158       0.0263158                  160.526   160.526            0.0263158
2        0.020202                    0.988129           2.60526    2.60526            1                0.988723   1                           0.989662            0.0263158       0.0526316                  160.526   160.526            0.0526316
3        0.030303                    0.986956           2.60526    2.60526            1                0.98764    1                           0.988988            0.0263158       0.0789474                  160.526   160.526            0.0789474
4        0.040404                    0.986253           2.60526    2.60526            1                0.986592   1                           0.988389            0.0263158       0.105263                   160.526   160.526            0.105263
5        0.0505051                   0.983891           2.60526    2.60526            1                0.984805   1                           0.987672            0.02631

In [30]:
best_model.accuracy()

[[0.4818895224478181, 0.9292929292929293]]

In [31]:
preds = best_model.predict(test_df_h2o)
preds.head()

gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


predict,p0,p1
0,0.966056,0.0339441
0,0.841551,0.158449
0,0.832241,0.167759
1,0.497027,0.502973
0,0.596389,0.403611
0,0.915552,0.0844483
0,0.678888,0.321112
0,0.971505,0.0284949
1,0.310666,0.689334
0,0.921239,0.0787613


In [32]:
pred_df = preds.as_data_frame()

submit = pd.DataFrame()
submit['PassengerId'] = df_test['PassengerId']
submit['Survived'] = pred_df['predict']
submit.to_csv('h2o.csv', sep=',', index=False)

C:\Users\admin\anaconda3\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


# CatBoost

https://tech.yandex.com/catboost/doc/dg/concepts/algorithm-main-stages_cat-to-numberic-docpage/  
https://arxiv.org/pdf/1706.09516.pdf  


In [21]:
from catboost import CatBoostClassifier
param_grid = {
    'iterations': [2, 3, 4, 5],
    'depth': [2, 3, 4, 5],
    'learning_rate': [1, 0.1, 0.01, 0.001]
}
cbm = randomized_cv(CatBoostClassifier(), param_grid)

0:	learn: 0.6564361	total: 134ms	remaining: 536ms
1:	learn: 0.6242013	total: 135ms	remaining: 202ms
2:	learn: 0.5994318	total: 135ms	remaining: 90.2ms
3:	learn: 0.5803730	total: 136ms	remaining: 33.9ms
4:	learn: 0.5600442	total: 136ms	remaining: 0us
0:	learn: 0.6534576	total: 410us	remaining: 1.64ms
1:	learn: 0.6223575	total: 824us	remaining: 1.24ms
2:	learn: 0.5957056	total: 1.26ms	remaining: 839us
3:	learn: 0.5760312	total: 1.6ms	remaining: 400us
4:	learn: 0.5559443	total: 1.91ms	remaining: 0us
0:	learn: 0.6541084	total: 410us	remaining: 1.64ms
1:	learn: 0.6228131	total: 758us	remaining: 1.14ms
2:	learn: 0.5973471	total: 1.1ms	remaining: 737us
3:	learn: 0.5775642	total: 1.43ms	remaining: 357us
4:	learn: 0.5586422	total: 1.75ms	remaining: 0us
0:	learn: 0.6521227	total: 397us	remaining: 1.59ms
1:	learn: 0.6199635	total: 736us	remaining: 1.1ms
2:	learn: 0.5911196	total: 1.06ms	remaining: 706us
3:	learn: 0.5716149	total: 1.38ms	remaining: 344us
4:	learn: 0.5520262	total: 1.76ms	remaining

Параметры модели

https://tech.yandex.com/catboost/doc/dg/concepts/python-reference_parameters-list-docpage/

Настройка параметров

https://tech.yandex.com/catboost/doc/dg/concepts/parameter-tuning-docpage/

Особенности

* уменьшено (?) переобучение
* умеет обрабатывать категориальные признаки
* большое количество визуализаций
* работает лучше по бенчмаркам (но дольше)

# Подбор гиперпараметров. Общий подход.

* выбрать относительно высокий learning_rate (например, 0.05 - 0.2)
* определить необходимое количество деревьев для исключения проблема недообучения и переобучения - поставить побольше и выбрать такое, где ошибка на валидации начинает расти
* зафиксировать параметры из предыдущих пунктов и настроить параметры, связанные с деревьями.
* зафиксировать параметры деревьев и дополнительно настроить learning_rate и количество деревьев

Основные параметры, связанные с бустингом

* learning_rate
* n_estimators
* subsample
* loss

Основные параметры, связанные с деревьями

* max_depth
* max_features
* min_samples_split
* min_samples_leaf
* max_leaf_nodes
* ...

In [22]:
import os
data_path = "/media/d_2000/data/hcdr/"
train = os.path.join(data_path, "application_train.csv")
test = os.path.join(data_path, "application_test.csv")

In [23]:
df = pd.read_csv(train)

FileNotFoundError: [Errno 2] No such file or directory: '/media/d_2000/data/hcdr/application_train.csv'